In [2]:
from pathlib import Path
import re

# file name patter found in /workspaces/3bc/stats_output/
file_name = f"dim%DIM%_objs%OBJ%_tree_%mode%.csv"

List the generations in a given experiment

In [3]:
def find_generations(experiment_dir: Path) -> list[int]:
    folders = [p.name for p in experiment_dir.iterdir() if p.is_dir()]
    generations: list[int] = [int(name.split("_", 1)[1]) for name in folders if name.startswith("gen_")]
    generations.sort()
    return generations


Find pairs $(n, m)$, where $n$ is the dimension of the decision space and $m$ is the number of objectives.

In [4]:

pattern = file_name.replace(r"%DIM%", r"(?P<dim>\d+)").replace(r"%OBJ%", r"(?P<obj>\d+)").replace(r"%mode%", r"(?P<mode>\w+)")

PAT = re.compile(pattern)

def find_pairs(experiment_dir: Path, generation: int) -> list[tuple[int, int, str]]:
    dirpath: Path = experiment_dir / f"gen_{generation}"
    pairs = set()
    for p in dirpath.iterdir():
        if not p.is_file():
            continue
        match = PAT.search(p.name)
        assert match, f"Expected file name {p.name} to match pattern {PAT.pattern}"
        pairs.add((
            int(match.group("dim")),
            int(match.group("obj")),
            match.group("mode"),
        ))
    return sorted(pairs)


Load dataframes

In [5]:
import polars as pl

def read_file(experiment_dir: Path, generation, codim, objective_dim, mode) -> pl.DataFrame:
    f = experiment_dir / f"gen_{generation}" / file_name.replace("%DIM%", str(codim)).replace("%OBJ%", str(objective_dim)).replace("%mode%", mode)
    df = pl.read_csv(f)
    return df.with_columns(
        pl.lit(str(experiment_dir)).alias("experiment"),
        pl.lit(generation).alias("generation"),
        pl.lit(codim).alias("codim"),
        pl.lit(objective_dim).alias("objective dim"),
        pl.lit(mode).alias("mode"),
    )


Analyze a particular experiment

In [6]:
experiment_dir = Path("/workspaces/3bc/stats_output/2026-05-04T13:13:23+08:00")
# experiment_dir = Path("/workspaces/3bc/stats_output/2026-05-01T12:37:06+00:00")
generation_all = find_generations(experiment_dir)
generation_final = generation_all[-1]

parameters = [
        (experiment_dir, generation_final) + params
        for params in find_pairs(experiment_dir=experiment_dir, generation=generation_final)
]

df = pl.concat([
    read_file(experiment_dir=exp, generation=gen, codim=codim, objective_dim=obj, mode=mode)
    for exp, gen, codim, obj, mode in parameters
], how="vertical")

dimensions = ["root", "node_1", "node_2", "node_3", "node_4"]

solvers = [
    "IBEA",    # indicator-based (hypervolume-based: getting in a sheet improves hypervolume)
    "MOEAD",   # decomposition-based (uniformly distributed sampling points in the objective space)
    "NSGAII",  # dominance-based (dominant solutions are selected for the next generation); inefficient for un-noisy problems
               # crowding distance (not based on volume but distance; 目的数が増えると crowding distance でタイブレークするが、体積に基づいてないのでおかしくなる。 rank 1 で埋まってる？) is used to maintain diversity in the population. Swap NSGA-II with NSGA-III?

    # "OMOPSO",
    # "GDE3"
    ]

# TODO: reduce design dim...
solver_name = {
    "IBEA": "IBEA",
    "MOEAD": "MOEA/D",
    "NSGAII": "NSGA-II",
    "OMOPSO": "OMOPSO",
    "GDE3": "GDE3"
}

# Rescale to log scale, and clip to avoid -inf values: 0 becomes 1e-1, and then log10(1e-1) = -1.
df = df.filter(pl.col("solver").is_in(solvers)) \
       .with_columns([
                pl.col(col).cast(pl.Float64).clip(lower_bound=1e-1).log10().alias(col)
                for col in dimensions])

In [7]:
df

,solver,exp_index,root,node_1,node_2,node_3,node_4,experiment,generation,codim,objective dim,mode
i64,str,i64,f64,f64,f64,f64,f64,str,i32,i32,i32,str
0,"""MOEAD""",1800,0.90309,-1.0,-1.0,-1.0,1.963788,"""/workspaces/3bc/stats_output/2…",200,2,2,"""breadth"""
1,"""MOEAD""",1801,0.477121,-1.0,-1.0,1.986772,-1.0,"""/workspaces/3bc/stats_output/2…",200,2,2,"""breadth"""
2,"""MOEAD""",1802,0.30103,-1.0,-1.0,-1.0,1.991226,"""/workspaces/3bc/stats_output/2…",200,2,2,"""breadth"""
3,"""MOEAD""",1803,0.90309,-1.0,-1.0,1.963788,-1.0,"""/workspaces/3bc/stats_output/2…",200,2,2,"""breadth"""
4,"""MOEAD""",1804,0.778151,-1.0,-1.0,-1.0,1.973128,"""/workspaces/3bc/stats_output/2…",200,2,2,"""breadth"""
…,…,…,…,…,…,…,…,…,…,…,…,…
495,"""IBEA""",3195,2.0,-1.0,-1.0,-1.0,-1.0,"""/workspaces/3bc/stats_output/2…",200,5,5,"""depth"""
496,"""IBEA""",3196,1.986772,0.477121,-1.0,-1.0,-1.0,"""/workspaces/3bc/stats_output/2…",200,5,5,"""depth"""
497,"""IBEA""",3197,2.0,-1.0,-1.0,-1.0,-1.0,"""/workspaces/3bc/stats_output/2…",200,5,5,"""depth"""


In [8]:
import plotly.express as px
import plotly.graph_objects as go

# solvers = sorted(df["solver"].unique().to_list())  # polars Series -> list of strings

colors = px.colors.qualitative.Safe
solver_to_color_id = {sol: i for i, sol in enumerate(solvers)}
solver_to_color    = {sol: colors[i % len(colors)] for i, sol in enumerate(solvers)}


def set_color(len_solvers: int):
    assert len_solvers <= len(colors), f"Not enough colors in palette for {len_solvers} solvers"

    colorscale = []
    for i in range(len_solvers):
        frac1 = i / len_solvers
        frac2 = (i + 1) / len_solvers
        colorscale.extend([[frac1, colors[i % len(colors)]], [frac2, colors[i % len(colors)]]])

    return {
        'color_continuous_scale': colorscale,
        'color_continuous_midpoint': 0.5 / len_solvers,
        'range_color': [0, len_solvers - 1],
    }

for mode, codim, objective_dim in [
    ("depth", 5, 2),
    ("depth", 4, 3),
    ("depth", 3, 4),
    ("depth", 2, 5),
]:

    fig = px.parallel_coordinates(
        (df_in_fig := df.filter(pl.col("mode") == mode) \
              .filter(pl.col("codim") == codim) \
              .filter(pl.col("objective dim") == objective_dim) \
              .with_columns(
                [pl.col("solver").replace_strict(solver_to_color_id, return_dtype=pl.Int64).alias("category_num")]
                )),
        dimensions=dimensions,
        color='category_num',
        title=f"Experiment {experiment_dir.name} - Generation {generation_final} - codim={codim}, objective dim={objective_dim}, mode={mode}",

        **set_color(len(solvers))
    )

    fig.update_traces(
        dimensions=[
            {
                "range": [-1,
                          df_in_fig[dimensions].to_numpy().max() # The max value across all dimensions
                          ],
            }
            for dim in dimensions
        ]
    )
    fig.update_layout(coloraxis_showscale=False)

    for i, (solver, color) in enumerate(solver_to_color.items()):
        fig.add_trace(go.Scatter(x=[None], y=[None],  # No data points
                                 mode='markers',
                                 marker=dict(color=color, size=10),
                                 name=solver_name[solver],
                                 showlegend=True,
                                 legendgroup=f'custom_{i}'))
        fig.update_layout(
            xaxis=dict(visible=False, showgrid=False, showticklabels=False),  # Hide x-axis
            yaxis=dict(visible=False, showgrid=False, showticklabels=False),  # Hide y-axis
            plot_bgcolor='rgba(0,0,0,0)',  # Transparent plot background
            paper_bgcolor='rgba(0,0,0,0)'   # Transparent paper background
        )

    fig.show()
